# Create positions from a Steve mosaic

Derives `boundary_positions*.txt`/`hole*.txt` automatically from a low-mag
Steve mosaic (e.g. `data/mosaic10x/`), instead of drawing them by hand.
Writes into `positions/` in the exact convention
`02_create_positions_from_boundaries.ipynb` already expects, so that notebook
picks the result up unchanged -- run this notebook first, then that one.

**Method:** the mosaic's tiles (each already stage-position-tagged by Steve)
are pasted into one flattened image; the image is smoothed, thresholded, and
cleaned up morphologically (closing to bridge small real gaps, opening to
drop noise specks, then dilated outward by a small margin) to get a tissue
mask; enclosed background regions inside the mask become holes. Both are
traced into polygons in real stage-micron coordinates -- see
`MERci.acquisition.mosaic` for the full pipeline.

**This is a visual, iterative notebook, not a one-shot script.** Re-run the
segmentation + review cells with different threshold/morphology parameters
until the green (tissue) / red (hole) overlay in the review plot looks right,
*then* run the final write cell -- it does not run automatically.

In [ ]:
import os
import sys
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity
from MERci.acquisition.mosaic import (
    load_steve_mosaic, assemble_mosaic_canvas, segment_mosaic_tissue,
    plot_mosaic_segmentation, save_boundary_from_mosaic,
)

POSITIONS_DIR = SAMPLE_DIR / "positions"
POSITIONS_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_NAME is the TRUE top-level experiment id -- see notebook 02's own
# docstring (02_create_positions_from_boundaries.ipynb) for why this isn't
# SAMPLE_DIR.name. Not used for naming here (this notebook only writes
# boundary_positions*.txt/hole*.txt, which carry no sample name), but printed
# for confirmation that MERCI_DIR resolves to the experiment you expect.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR    : {SAMPLE_DIR}")
print(f"SAMPLE_NAME   : {SAMPLE_NAME}")
print(f"POSITIONS_DIR : {POSITIONS_DIR}")

## Load the mosaic

`MOSAIC_DIR` defaults to `SAMPLE_DIR/data/mosaic10x` -- the folder
`02_create_positions_from_boundaries.ipynb` already creates as a placeholder,
and where Steve's own "Save Mosaic" writes its `.msc` manifest + `.stv` tile
files. Set `MOSAIC_DIR`/`MOSAIC_NAME` manually if yours lives elsewhere or is
named differently.

In [ ]:
MOSAIC_DIR  = SAMPLE_DIR / "data" / "mosaic10x"
MOSAIC_NAME = None   # e.g. "mosaic10x" -- None = auto-detect the only *.msc file present

msc_candidates = sorted(MOSAIC_DIR.glob(f"{MOSAIC_NAME or '*'}.msc"))
if not msc_candidates:
    raise FileNotFoundError(
        f"No .msc mosaic manifest found in {MOSAIC_DIR} -- copy Steve's saved "
        f"mosaic there first (or set MOSAIC_DIR/MOSAIC_NAME above)."
    )
if len(msc_candidates) > 1:
    print(f"WARNING: {len(msc_candidates)} .msc files found, using the first: "
          f"{msc_candidates[0].name}. Set MOSAIC_NAME to pick a different one.")
MSC_PATH = msc_candidates[0]

tiles = load_steve_mosaic(MSC_PATH)
print(f"Loaded {len(tiles)} tile(s) from {MSC_PATH.name}")
print(f"Tile pixel size: {tiles[0].pixel_size_um:.4f} um/px  "
      f"({tiles[0].image.shape[1]}x{tiles[0].image.shape[0]} px/tile)")

WORKING_PIXEL_UM = 5.0   # canvas resolution -- smaller = sharper but slower/more memory
canvas = assemble_mosaic_canvas(tiles, working_pixel_um=WORKING_PIXEL_UM)
print(f"Canvas: {canvas.image.shape[1]}x{canvas.image.shape[0]} px "
      f"at {canvas.pixel_size_um:.2f} um/px, origin {canvas.origin_um}")

## Segment tissue + holes -- tune and re-run

Start with the defaults; re-run this cell and the plot cell below with
adjusted parameters until the overlay looks right. `THRESHOLD = None` uses
an automatic (Otsu) threshold on the smoothed canvas -- set a fixed number
to override it once you've seen the canvas's intensity range. All
distances are in real microns, not canvas pixels, so they carry over even if
you change `WORKING_PIXEL_UM` above.

- `SMOOTH_SIGMA_UM` -- Gaussian blur before thresholding, to suppress
  per-tile illumination speckle. Too small: noisy/fragmented mask. Too
  large: blurs away real fine structure.
- `CLOSE_RADIUS_UM` -- bridges small real gaps between adjacent bits of the
  same tissue piece so they merge into one polygon instead of many.
- `OPEN_RADIUS_UM` -- removes small noise specks that survive closing.
- `MARGIN_UM` -- outward safety buffer applied after cleanup (a hand-drawn
  boundary naturally includes some margin beyond the exact signal edge).
- `MIN_TISSUE_AREA_UM2`/`MIN_HOLE_AREA_UM2` -- drop components smaller than
  this after morphology.
- `SIMPLIFY_TOL_UM` -- polygon simplification tolerance (marching squares
  otherwise emits one vertex per canvas pixel of perimeter).

In [ ]:
THRESHOLD           = None   # None = auto (Otsu on the smoothed canvas)
SMOOTH_SIGMA_UM     = 10.0
CLOSE_RADIUS_UM     = 50.0
OPEN_RADIUS_UM      = 15.0
MARGIN_UM           = 75.0
MIN_TISSUE_AREA_UM2 = 1000.0
MIN_HOLE_AREA_UM2   = 500.0
SIMPLIFY_TOL_UM     = 15.0

segmentation = segment_mosaic_tissue(
    canvas,
    threshold           = THRESHOLD,
    smooth_sigma_um     = SMOOTH_SIGMA_UM,
    close_radius_um     = CLOSE_RADIUS_UM,
    open_radius_um      = OPEN_RADIUS_UM,
    margin_um           = MARGIN_UM,
    min_tissue_area_um2 = MIN_TISSUE_AREA_UM2,
    min_hole_area_um2   = MIN_HOLE_AREA_UM2,
    simplify_tol_um     = SIMPLIFY_TOL_UM,
)
print(f"Threshold used : {segmentation.threshold:.1f}")
print(f"Tissue pieces  : {len(segmentation.tissue_polygons)}")
print(f"Holes          : {len(segmentation.hole_polygons)}")
for i, p in enumerate(sorted(segmentation.tissue_polygons, key=lambda p: -p.area)):
    print(f"  tissue[{i}] area={p.area:,.0f} um^2")

In [ ]:
ax = plot_mosaic_segmentation(canvas, segmentation)
ax.figure.set_size_inches(10, 10)
ax.figure.tight_layout()

## Write boundary_positions*.txt / hole*.txt

Only run this once the plot above looks right. A single detected tissue
piece is written as the legacy `boundary_positions.txt`; several disjoint
pieces are written as `boundary_positions_{b}.txt` (the "single" layout --
see `discover_boundary_files`). Existing files with the same name are
overwritten, so re-running after a parameter change is safe.

After this, continue with `02_create_positions_from_boundaries.ipynb`.

In [ ]:
written = save_boundary_from_mosaic(segmentation, POSITIONS_DIR)
print(f"Wrote {len(written)} file(s) to {POSITIONS_DIR}:")
for f in written:
    print(f"  {f}")